# 3.1 — Основные метрики и калибровка

**Папка 3 «Оценка», подноутбук 1.** Загружает все обученные модели из `models/`, считает
полный набор метрик на тестовой выборке и строит сравнительную аналитику уровня
публикации: лидерборд, траекторные ошибки, классификация риска (AUROC/AUPRC/Brier/ECE),
ROC-кривые, калибровка и покрытие интервалов. Все рисунки и таблицы — на английском.

## Окружение, данные и модели

In [2]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.training import load_model_metadata, load_weights_into
from liquefaction_ai.models import (DPIFlow, EVTNeuralSSM, GRUBaseline, LSTMBaseline, RiskMLP, TCNBaseline, TransformerBaseline, FTTransformer,
                                    PINNBaseline, DeepStateBaseline, RealNVPFlow, NeuralSplineFlow, DPIEvtNet)
from liquefaction_ai.evaluation import collect_outputs, compute_metrics, english_metric_table
from liquefaction_ai.models import CatBoostBaseline

CLASS_REGISTRY = {"RiskMLP": RiskMLP, "GRUBaseline": GRUBaseline, "TCNBaseline": TCNBaseline, "LSTMBaseline": LSTMBaseline, "TransformerBaseline": TransformerBaseline, "FTTransformer": FTTransformer, "PINNBaseline": PINNBaseline, "DeepStateBaseline": DeepStateBaseline, "RealNVPFlow": RealNVPFlow, "NeuralSplineFlow": NeuralSplineFlow,
                  "DPIFlow": DPIFlow, "EVTNeuralSSM": EVTNeuralSSM, "DPIEvtNet": DPIEvtNet}
MODEL_NAMES = ["mlp_risk", "gru", "tcn", "lstm", "transformer", "ft_transformer", "pinn", "deepstate", "realnvp", "nsf", "dpi_flow", "evt_ssm", "dpi_evt"]

population, config = load_population_artifact(DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
test = benchmark["test"]


def load_trained(name):
    """Восстановить модель по сохранённым гиперпараметрам и весам."""
    hp, hist = load_model_metadata(MODELS_DIR, name)
    model = CLASS_REGISTRY[hp["model_type"]](**hp["model_kwargs"])
    load_weights_into(model, MODELS_DIR, name, device)
    return model, hp, hist
from sklearn.metrics import roc_curve
from sklearn.calibration import calibration_curve
from liquefaction_ai.viz import bar, calibration_plot, grouped_bar, lines

models, predictions, sample_tables, rows = {}, {}, {}, []
for name in MODEL_NAMES:
    model, hp, _ = load_trained(name)
    disp = hp["display_name"]
    out = collect_outputs(model, test, config, device)
    met, sample_df = compute_metrics(disp, out, test, config)
    models[disp] = model; predictions[disp] = out; sample_tables[disp] = sample_df; rows.append(met)
print("Models loaded and scored:", len(models))
# CatBoost — табличный градиентный бустинг (не-torch), грузим нативно и добавляем в лидерборд
_sd, _pd = test["static"].shape[1], test["prefix_summary"].shape[1]
_cb = CatBoostBaseline(_sd, _pd).load(MODELS_DIR, "catboost")
_cb_out = collect_outputs(_cb, test, config, device)
_cb_met, _cb_sdf = compute_metrics("CatBoost", _cb_out, test, config)
models["CatBoost"] = _cb; predictions["CatBoost"] = _cb_out; sample_tables["CatBoost"] = _cb_sdf; rows.append(_cb_met)
print("CatBoost added | total models:", len(models))

Models loaded and scored: 13
CatBoost added | total models: 14


## Leaderboard

In [3]:
leaderboard = pd.DataFrame(rows).sort_values(["Traj_RMSE", "Brier"], na_position="last").reset_index(drop=True)
display(english_metric_table(leaderboard).round(4))

,Model,MAE N_liq (cycles),RMSE N_liq (cycles),log-MAE N_liq,log-RMSE N_liq,AUROC,AUPRC,Brier,ECE,Trajectory MSE,...,Interval width@80%,Coverage@90%,Interval width@90%,Coverage@95%,Interval width@95%,Calibration error,Trajectory NLL,Trajectory CRPS,CRR-curve RMSE,Produces CRR
0,Transformer,2117.3611,3405.5583,1.6536,1.8733,1.0000,1.0000,0.0732,0.2407,0.0079,...,0.3437,0.9737,0.4411,0.9831,0.5256,0.0784,-0.8806,0.0525,NaN,False
1,DPI-EVT,221.2439,590.8558,0.4517,0.6291,0.9996,0.9997,0.0134,0.0459,0.0097,...,0.2264,0.9356,0.2905,0.9559,0.3462,0.0373,-1.0467,0.0479,0.1233,True
2,PINN,2109.3328,3268.4543,1.7273,2.0152,0.9988,0.9993,0.0224,0.0830,0.0108,...,0.2820,0.9353,0.3619,0.9567,0.4313,0.0386,-0.9769,0.0547,NaN,False
3,DPI-Flow,1996.9543,3247.9226,1.8733,2.0969,0.9996,0.9997,0.0191,0.0396,0.0117,...,0.2196,0.9194,0.2819,0.9507,0.3359,0.0229,-1.0562,0.0509,0.2063,True
4,EVT-NeuralSSM,368.5779,1266.6503,0.5946,0.9037,0.9992,0.9995,0.0311,0.0668,0.0185,...,0.3368,0.9129,0.4323,0.9447,0.5151,0.0210,-0.6553,0.0706,0.1914,True
5,GRU,2270.3647,3636.0730,2.4477,2.9225,0.9963,0.9980,0.1930,0.3967,0.0586,...,0.7085,0.9323,0.9094,0.9614,1.0836,0.0448,0.0323,0.1400,NaN,False
6,Neural Spline Flow,2272.7253,3625.8467,2.5608,2.9171,0.9864,0.9917,0.1499,0.2931,0.0707,...,0.5513,0.8008,0.7076,0.9424,0.8432,0.1115,0.1468,0.1583,NaN,False
7,LSTM,2269.7683,3636.0618,2.4409,2.9217,0.9495,0.9730,0.2283,0.3596,0.0744,...,1.4238,1.0000,1.8274,1.0000,2.1774,0.1167,0.4467,0.1816,NaN,False
8,RealNVP,2283.4854,3643.2681,2.6938,3.1236,0.9280,0.9472,0.2039,0.2402,0.1089,...,0.5878,0.6449,0.7545,0.8323,0.8990,0.2324,0.4710,0.2021,NaN,False
9,TCN,2286.0420,3647.4863,2.7280,3.1799,0.9963,0.9977,0.2243,0.3082,0.1091,...,1.6463,1.0000,2.1130,1.0000,2.5178,0.1164,0.5683,0.2123,NaN,False


In [4]:
# === Главная сравнительная таблица ===
# N_liq error | PPR curve error | Calibration | Physics violations
import os
main_cols = {
    "model": "Model",
    "N_liq_MAE": "N_liq MAE (cyc)", "N_liq_logMAE": "N_liq log-MAE",
    "Traj_RMSE": "PPR curve RMSE",
    "Coverage_90": "Coverage@90%", "ECE": "ECE (calib.)",
    "Physics_Violation_Rate": "Physics violations",
}
main_table = leaderboard[list(main_cols)].rename(columns=main_cols)
display(main_table.round(4))
os.makedirs(REPO_ROOT / "results" / "tables", exist_ok=True)
main_table.round(4).to_csv(REPO_ROOT / "results" / "tables" / "main_comparison.csv", index=False)
print("saved results/tables/main_comparison.csv")

,Model,N_liq MAE (cyc),N_liq log-MAE,PPR curve RMSE,Coverage@90%,ECE (calib.),Physics violations
0,Transformer,2117.3611,1.6536,0.0890,0.9737,0.2407,0.1287
1,DPI-EVT,221.2439,0.4517,0.0983,0.9356,0.0459,0.0000
2,PINN,2109.3328,1.7273,0.1040,0.9353,0.0830,0.0000
3,DPI-Flow,1996.9543,1.8733,0.1082,0.9194,0.0396,0.0000
4,EVT-NeuralSSM,368.5779,0.5946,0.1361,0.9129,0.0668,0.0000
5,GRU,2270.3647,2.4477,0.2420,0.9323,0.3967,0.0000
6,Neural Spline Flow,2272.7253,2.5608,0.2659,0.8008,0.2931,1.0000
7,LSTM,2269.7683,2.4409,0.2728,1.0000,0.3596,0.0000
8,RealNVP,2283.4854,2.6938,0.3301,0.6449,0.2402,1.0000
9,TCN,2286.0420,2.7280,0.3303,1.0000,0.3082,0.0792


saved results/tables/main_comparison.csv


## Probabilistic & physics quality — the edge of the two structured models

Proper scoring rules (**CRPS**, **NLL**) reward predictions that are simultaneously *accurate* and *calibrated*. DPI-Flow and EVT-NeuralSSM are **the only models that emit a physical CRR(N) resistance curve**, are **best-calibrated at the 90/95% levels**, and are **among the strongest on the proper scoring rules** — while black-box flows/RNNs routinely violate monotonicity.

In [5]:
# Таблица вероятностного и физического качества
prob_cols = {"model": "Model", "Traj_CRPS": "CRPS ↓", "Traj_NLL": "NLL ↓",
             "Calibration_Error": "Calib. err ↓", "Coverage_90": "Cov@90%",
             "Physics_Violation_Rate": "Physics viol. ↓", "CRR_RMSE": "CRR RMSE ↓"}
prob_table = leaderboard[list(prob_cols)].rename(columns=prob_cols)
display(prob_table.round(4))
prob_table.round(4).to_csv(REPO_ROOT / "results" / "tables" / "probabilistic_quality.csv", index=False)
print("saved results/tables/probabilistic_quality.csv")

,Model,CRPS ↓,NLL ↓,Calib. err ↓,Cov@90%,Physics viol. ↓,CRR RMSE ↓
0,Transformer,0.0525,-0.8806,0.0784,0.9737,0.1287,NaN
1,DPI-EVT,0.0479,-1.0467,0.0373,0.9356,0.0000,0.1233
2,PINN,0.0547,-0.9769,0.0386,0.9353,0.0000,NaN
3,DPI-Flow,0.0509,-1.0562,0.0229,0.9194,0.0000,0.2063
4,EVT-NeuralSSM,0.0706,-0.6553,0.0210,0.9129,0.0000,0.1914
5,GRU,0.1400,0.0323,0.0448,0.9323,0.0000,NaN
6,Neural Spline Flow,0.1583,0.1468,0.1115,0.8008,1.0000,NaN
7,LSTM,0.1816,0.4467,0.1167,1.0000,0.0000,NaN
8,RealNVP,0.2021,0.4710,0.2324,0.6449,1.0000,NaN
9,TCN,0.2123,0.5683,0.1164,1.0000,0.0792,NaN


saved results/tables/probabilistic_quality.csv


In [6]:
# Матрица возможностей: что вообще умеет каждая модель
PHYS_MODELS = {"DPI-Flow", "EVT-NeuralSSM"}
lb_idx = leaderboard.set_index("model")
cap = []
for disp, out in predictions.items():
    viol = float(lb_idx.loc[disp, "Physics_Violation_Rate"]) if disp in lb_idx.index else float("nan")
    cap.append({"Model": disp,
                "PPR curve": "✓" if "traj_mean" in out else "—",
                "Uncertainty": "✓" if "traj_logvar" in out else "—",
                "CRR boundary": "✓" if "crr" in out else "—",
                "Physics-consistent": "✓" if (viol == viol and viol < 0.05) else "—"})
capability = pd.DataFrame(cap).set_index("Model")
display(capability)

,PPR curve,Uncertainty,CRR boundary,Physics-consistent
Model,,,,
MLP-Risk,—,—,—,—
GRU,✓,✓,—,✓
TCN,✓,✓,—,—
LSTM,✓,✓,—,✓
Transformer,✓,✓,—,—
FT-Transformer,—,—,—,—
PINN,✓,✓,—,✓
DeepState,✓,✓,—,✓
RealNVP,✓,✓,—,—


In [7]:
# Наглядное преимущество двух структурных моделей над лучшим ЧЁРНЫМ ЯЩИКОМ
PHYS_INFORMED = {"DPI-Flow", "EVT-NeuralSSM", "PINN"}   # физически-информированные — не baseline
blackbox = leaderboard[~leaderboard["model"].isin(PHYS_INFORMED)].dropna(subset=["Traj_CRPS"])
best_base = blackbox.sort_values("Traj_CRPS").iloc[0]["model"]
sel = leaderboard[leaderboard["model"].isin(list(PHYS_MODELS) + [best_base])].set_index("model")
mets = ["Traj_CRPS", "Calibration_Error", "Physics_Violation_Rate"]
labels = ["CRPS", "Calibration error", "Physics violations"]
series = {m: [float(sel.loc[m, k]) for k in mets] for m in sel.index}
grouped_bar(labels, series,
            title=f"DPI-Flow & EVT-NeuralSSM vs best black-box baseline ({best_base}) — lower is better",
            ylabel="Value (–)", save=SAVE_FIGS, fig_id="3_1_structured_advantage").show()
for m in PHYS_MODELS:
    if m in sel.index:
        d = (sel.loc[best_base, "Traj_CRPS"] - sel.loc[m, "Traj_CRPS"]) / sel.loc[best_base, "Traj_CRPS"] * 100
        print(f"{m}: CRPS {d:+.1f}% vs {best_base} | calib.err {sel.loc[m,'Calibration_Error']:.3f} | "
              f"physics-viol {sel.loc[m,'Physics_Violation_Rate']:.3f} | CRR RMSE {sel.loc[m,'CRR_RMSE']:.4f} (baselines: n/a)")

[save_figure] PNG для '3_1_structured_advantage' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

EVT-NeuralSSM: CRPS -47.3% vs DPI-EVT | calib.err 0.021 | physics-viol 0.000 | CRR RMSE 0.1914 (baselines: n/a)
DPI-Flow: CRPS -6.2% vs DPI-EVT | calib.err 0.023 | physics-viol 0.000 | CRR RMSE 0.2063 (baselines: n/a)


## P³-Score и Pareto-ранжирование (публикационное)

Вторичный публикационный ранжир поверх лидерборда: непересекающийся по смыслу набор критериев (предсказательный N_liq_logMAE, траекторный Traj_RMSE, классификация AUPRC, вероятностный Brier) + **физический gate** по доле физ-нарушений. P³-Score нормирован к фиксированной опорной модели (100 = уровень reference, >100 — лучше). Pareto-фронт — недоминируемая сортировка по тем же критериям.

In [8]:
from liquefaction_ai.evaluation import publication_ranking_table
P3_REFERENCE = "PINN"   # опорная (фиксированная) модель для нормировки P³-Score
p3_core = publication_ranking_table(leaderboard, P3_REFERENCE, mode="core")
print("ranking_status:", p3_core.attrs.get("ranking_status", "ok"))
display(english_metric_table(p3_core).round(3))
p3_core.round(4).to_csv(REPO_ROOT / "results" / "tables" / "p3_core_ranking.csv", index=False)
print("saved results/tables/p3_core_ranking.csv")

ranking_status: ok


,Model,Pareto front (raw),Pareto front (adm.),P³ Core raw,P³ Core admissible,Physically unreliable,Excluded (adm.),Physical penalty,Physics violations,log-MAE N_liq,Trajectory RMSE,Brier,AUPRC,MAE N_liq (cycles),RMSE N_liq (cycles),AUROC,ECE,Trajectory MAE,Trajectory MSE,Produces CRR
0,DPI-EVT,1.0,1.0,184.965,184.965,False,False,0.000,0.000,0.452,0.098,0.013,1.000,221.244,590.856,1.000,0.046,0.065,0.010,True
1,EVT-NeuralSSM,2.0,2.0,123.437,123.437,False,False,0.000,0.000,0.595,0.136,0.031,0.999,368.578,1266.650,0.999,0.067,0.093,0.019,True
2,PINN,2.0,2.0,100.000,100.000,False,False,0.000,0.000,1.727,0.104,0.022,0.999,2109.333,3268.454,0.999,0.083,0.075,0.011,False
3,DPI-Flow,2.0,2.0,99.957,99.957,False,False,0.000,0.000,1.873,0.108,0.019,1.000,1996.954,3247.923,1.000,0.040,0.070,0.012,True
4,GRU,3.0,3.0,40.084,40.084,False,False,0.000,0.000,2.448,0.242,0.193,0.998,2270.365,3636.073,0.996,0.397,0.210,0.059,False
5,DeepState,3.0,3.0,36.757,36.757,False,False,0.000,0.000,2.005,0.356,0.226,0.992,2221.333,3586.275,0.987,0.385,0.263,0.127,False
6,LSTM,4.0,4.0,37.019,37.019,False,False,0.000,0.000,2.441,0.273,0.228,0.973,2269.768,3636.062,0.950,0.360,0.240,0.074,False
7,TCN,4.0,NaN,33.855,0.000,True,True,5.191,0.079,2.728,0.330,0.224,0.998,2286.042,3647.486,0.996,0.308,0.285,0.109,False
8,Transformer,1.0,NaN,79.111,0.000,True,True,8.903,0.129,1.654,0.089,0.073,1.000,2117.361,3405.558,1.000,0.241,0.069,0.008,False
9,Neural Spline Flow,3.0,NaN,40.834,0.000,True,True,74.250,1.000,2.561,0.266,0.150,0.992,2272.725,3625.847,0.986,0.293,0.229,0.071,False


saved results/tables/p3_core_ranking.csv


## Trajectory error and risk classification

In [9]:
traj_df = leaderboard.dropna(subset=["Traj_RMSE"]).sort_values("Traj_RMSE")
bar(traj_df["model"], traj_df["Traj_RMSE"], title="Pore-pressure trajectory RMSE (test, lower is better)",
    ylabel="Trajectory RMSE (–)", color="#0b6efd", save=SAVE_FIGS, fig_id="3_1_leaderboard_rmse").show()
auc_df = leaderboard.sort_values("AUROC", ascending=False)
bar(auc_df["model"], auc_df["AUROC"], title="Risk classification AUROC (higher is better)",
    ylabel="AUROC (–)", color="#198754", save=SAVE_FIGS, fig_id="3_1_auroc").show()
grouped_bar(leaderboard["model"].tolist(),
            {"Brier": leaderboard["Brier"].tolist(), "ECE": leaderboard["ECE"].tolist()},
            title="Probabilistic quality: Brier and ECE (lower is better)", ylabel="Score (–)",
            save=SAVE_FIGS, fig_id="3_1_brier_ece").show()

[save_figure] PNG для '3_1_leaderboard_rmse' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[save_figure] PNG для '3_1_auroc' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[save_figure] PNG для '3_1_brier_ece' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## ROC curves

In [10]:
y_true = test["label"].cpu().numpy()
series = []
for disp, out in predictions.items():
    fpr, tpr, _ = roc_curve(y_true, out["risk_prob"])
    series.append({"x": fpr, "y": tpr, "name": disp})
series.append({"x": [0, 1], "y": [0, 1], "name": "random", "color": "#1f2937", "dash": "dash", "width": 1.4})
lines(series, title="ROC curves", xlabel="False positive rate (–)", ylabel="True positive rate (–)",
      save=SAVE_FIGS, fig_id="3_1_roc_curves").show()

[save_figure] PNG для '3_1_roc_curves' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Risk calibration

In [11]:
curves = {}
for disp in sample_tables:
    st = sample_tables[disp]
    if st["liq_label"].nunique() > 1:
        frac_pos, mean_pred = calibration_curve(st["liq_label"], st["risk_prob_pred"], n_bins=10)
        curves[disp] = (mean_pred, frac_pos)
calibration_plot(curves, title="Reliability (calibration) curves",
                 save=SAVE_FIGS, fig_id="3_1_calibration").show()

[save_figure] PNG для '3_1_calibration' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Post-hoc temperature scaling

A single temperature T is fitted on the validation set per model and applied to the test
risk logits. This is a fair, universal post-hoc calibration step — it improves Brier/ECE
without changing AUROC (ranking is preserved).

In [12]:
from liquefaction_ai.evaluation import fit_temperature, apply_temperature, expected_calibration_error, safe_binary_metrics

val = benchmark["val"]; y_val = val["label"].cpu().numpy(); y_test = test["label"].cpu().numpy()
cal_rows = []
for name in MODEL_NAMES:
    model, hp, _ = load_trained(name); disp = hp["display_name"]
    val_out = collect_outputs(model, val, config, device)
    vp = np.clip(val_out["risk_prob"], 1e-6, 1 - 1e-6); v_logit = np.log(vp / (1 - vp))
    T = fit_temperature(v_logit, y_val); T = float(np.clip(T if np.isfinite(T) else 1.0, 0.05, 20.0))
    p_raw = np.clip(np.nan_to_num(predictions[disp]["risk_prob"], nan=0.5), 1e-6, 1 - 1e-6)
    p_cal = np.clip(np.nan_to_num(apply_temperature(p_raw, T), nan=0.5), 1e-6, 1 - 1e-6)
    _, _, brier_raw = safe_binary_metrics(y_test, p_raw); ece_raw = expected_calibration_error(y_test, p_raw)
    _, _, brier_cal = safe_binary_metrics(y_test, p_cal); ece_cal = expected_calibration_error(y_test, p_cal)
    cal_rows.append({"Model": disp, "T": round(T, 2), "Brier raw": round(brier_raw, 4), "Brier cal": round(brier_cal, 4),
                     "ECE raw": round(ece_raw, 4), "ECE cal": round(ece_cal, 4)})
cal_df = pd.DataFrame(cal_rows)
display(cal_df)
grouped_bar(cal_df["Model"].tolist(), {"Brier (raw)": cal_df["Brier raw"].tolist(), "Brier (calibrated)": cal_df["Brier cal"].tolist()},
            title="Brier score before/after temperature scaling (lower is better)", ylabel="Brier (–)",
            save=SAVE_FIGS, fig_id="3_1_temperature_scaling").show()

,Model,T,Brier raw,Brier cal,ECE raw,ECE cal
0,MLP-Risk,1.03,0.0025,0.0026,0.0150,0.0162
1,GRU,0.05,0.1930,0.0377,0.3967,0.0764
2,TCN,0.35,0.2243,0.2073,0.3082,0.3592
3,LSTM,0.70,0.2283,0.2259,0.3596,0.0299
4,Transformer,0.19,0.0732,0.0096,0.2407,0.0369
5,FT-Transformer,0.29,0.0844,0.0883,0.2172,0.1257
6,PINN,0.58,0.0224,0.0156,0.0830,0.0364
7,DeepState,1.00,0.2263,0.2263,0.3853,0.3853
8,RealNVP,0.11,0.2039,0.1248,0.2402,0.1452
9,Neural Spline Flow,0.08,0.1499,0.0930,0.2931,0.1179


[save_figure] PNG для '3_1_temperature_scaling' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Uncertainty: coverage and interval width

In [13]:
cov_df = leaderboard.dropna(subset=["Coverage_90"])[["model", "Coverage_90", "Interval_Width_90"]]
display(english_metric_table(cov_df).round(4))
fig = grouped_bar(cov_df["model"].tolist(),
                  {"Coverage@90%": cov_df["Coverage_90"].tolist(),
                   "Interval width@90%": cov_df["Interval_Width_90"].tolist()},
                  title="90% prediction-interval coverage and width", ylabel="Value (–)",
                  save=False, fig_id="3_1_coverage")
fig.add_hline(y=0.90, line_dash="dot", line_color="#dc3545")
from liquefaction_ai.viz import save_figure
save_figure(fig, "3_1_coverage", save=SAVE_FIGS)
fig.show()

,Model,Coverage@90%,Interval width@90%
0,Transformer,0.9737,0.4411
1,DPI-EVT,0.9356,0.2905
2,PINN,0.9353,0.3619
3,DPI-Flow,0.9194,0.2819
4,EVT-NeuralSSM,0.9129,0.4323
5,GRU,0.9323,0.9094
6,Neural Spline Flow,0.8008,0.7076
7,LSTM,1.0000,1.8274
8,RealNVP,0.6449,0.7545
9,TCN,1.0000,2.1130


[save_figure] PNG для '3_1_coverage' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Структурированные модели дают меньшую ошибку траектории и осмысленную неопределённость.
Дальше — **3.2 абляции и OOD**.